# RUL prediction with an LSTM (OOP version)

| Pillar | Where it appears in this notebook |
|---|---|
| **Abstraction** | `BaseRULModel` / `BaseRULPreprocessor` (in `src/base.py`) define the contract. |
| **Encapsulation** | Private state: `_pipeline`, `_seq_length`, and the name-mangled `__history`, exposed via properties. |
| **Inheritance** | `SequencePreprocessor` extends `BaseRULPreprocessor`; `LSTMRULModel` extends `BaseRULModel`; sklearn transformers extend `BaseEstimator`/`TransformerMixin`. |
| **Polymorphism** | `LSTMRULModel` overrides the same methods as `XGBRULModel` with Keras-specific behaviour; the same `ExperimentRunner` drives both. |

In [1]:
import os
import random

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Dense, LSTM, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import Sequential
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

from src.handle_data import CMAPSSDataLoader
from src.base import BaseRULModel, BaseRULPreprocessor, ExperimentRunner, RegressionEvaluator

## 1. Reproducibility

In [2]:
class SeedManager:
    """Fixes every random seed so runs are reproducible."""

    @staticmethod
    def set_seed(seed=42):
        os.environ["PYTHONHASHSEED"] = "0"
        random.seed(seed)
        np.random.seed(seed)
        tf.random.set_seed(seed)


SeedManager.set_seed(42)

## 2. Helper classes (sklearn transformers, sequence builder, validator)

In [3]:
class RollingFeatureTransformer(BaseEstimator, TransformerMixin):
    """Adds per-engine rolling mean and std columns for the given sensors."""

    def __init__(self, sensor_cols, window=27):
        self.sensor_cols = sensor_cols
        self.window = window

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.sort_values(["engine_id", "cycle"]).copy()
        grouped = df.groupby("engine_id", sort=False)
        for col in self.sensor_cols:
            df[f"{col}_roll_mean"] = grouped[col].transform(
                lambda s: s.rolling(window=self.window, min_periods=1).mean())
            df[f"{col}_roll_std"] = grouped[col].transform(
                lambda s: s.rolling(window=self.window, min_periods=2).std())
        return df


class IdColumnDropper(BaseEstimator, TransformerMixin):
    """Removes the engine_id column before scaling."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop(columns=["engine_id"])


class SequenceBuilder:
    """Turns scaled per-cycle rows into fixed-length sliding windows, engine by engine."""

    def __init__(self, seq_length=30):
        self.seq_length = seq_length

    def build_train(self, df, X, y):
        X, y = np.asarray(X), np.asarray(y)
        engine_ids = df["engine_id"].to_numpy()
        sequences, targets = [], []
        for engine_id in df["engine_id"].unique():
            indices = np.flatnonzero(engine_ids == engine_id)
            engine_X = X[indices]
            if len(engine_X) < self.seq_length:      # skip very short engines
                continue
            for end in range(self.seq_length, len(engine_X) + 1):
                sequences.append(engine_X[end - self.seq_length:end])
                targets.append(y[indices[end - 1]])
        return np.asarray(sequences, dtype=np.float32), np.asarray(targets, dtype=np.float32)

    def build_test(self, df, X):
        """One window per test engine: its last `seq_length` cycles."""
        X = np.asarray(X)
        engine_ids = df["engine_id"].to_numpy()
        sequences = []
        for engine_id in df["engine_id"].unique():
            engine_X = X[np.flatnonzero(engine_ids == engine_id)]
            if len(engine_X) >= self.seq_length:
                sequences.append(engine_X[-self.seq_length:])
        return np.asarray(sequences, dtype=np.float32)


class DataValidator:
    """Sanity checks for NaN / infinity values."""

    @staticmethod
    def report(**arrays):
        for name, arr in arrays.items():
            arr = np.asarray(arr, dtype=float)
            print(f"{name}: shape={arr.shape}, NaNs={np.isnan(arr).sum()}, infs={np.isinf(arr).sum()}, "
                  f"range=({np.nanmin(arr):.3f}, {np.nanmax(arr):.3f})")

## 3. Concrete preprocessor (inherits cleaning logic from the base class)

In [4]:
class SequencePreprocessor(BaseRULPreprocessor):
    """Scaled, windowed sequences (samples, seq_length, features) for recurrent models."""

    def __init__(self, rul_cap=125, window=27, seq_length=30):
        super().__init__(rul_cap=rul_cap, window=window, clip_test_target=False)
        self._seq_length = seq_length
        self._builder = SequenceBuilder(seq_length)
        self._pipeline = Pipeline([
            ("rolling_features", RollingFeatureTransformer(self.SENSOR_COLS, self.window)),
            ("drop_id", IdColumnDropper()),
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", MinMaxScaler()),
        ])

    @property
    def seq_length(self):
        return self._seq_length

    def prepare(self, train_df, test_df, test_last):        # overrides the abstract method
        train_df, test_df = self._clean(train_df, test_df)

        X_train_df = train_df.drop(columns=["RUL"])
        y_train = train_df["RUL"].to_numpy(dtype=np.float32)

        X_train_scaled = self._pipeline.fit_transform(X_train_df)   # fit on training data ONLY
        X_test_scaled = self._pipeline.transform(test_df)

        X_train_seq, y_train_seq = self._builder.build_train(X_train_df, X_train_scaled, y_train)
        X_test_seq = self._builder.build_test(test_df, X_test_scaled)
        y_test = self._test_target(test_last)
        return X_train_seq, y_train_seq, X_test_seq, y_test

## 4. Concrete model (inherits fit / predict / evaluate from the base class)

In [5]:
class LSTMRULModel(BaseRULModel):
    """Single-layer LSTM regressor for remaining useful life."""

    def __init__(self, rul_cap=125, seq_length=30, units=64, learning_rate=1e-3):
        super().__init__(name="LSTM", rul_cap=rul_cap)
        self._seq_length = seq_length
        self._units = units
        self._learning_rate = learning_rate
        self.__history = None                               # truly private (name-mangled)

    @property
    def history(self):
        return self.__history

    def _build_model(self, X):
        n_features = X.shape[2]
        model = Sequential([
            Input(shape=(self._seq_length, n_features)),
            LSTM(self._units),
            Dense(1, activation="linear"),
        ])
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=self._learning_rate),
            loss="mse",
            metrics=[keras.metrics.MeanAbsoluteError(name="mae"),
                     keras.metrics.R2Score(name="r2_score")],
        )
        return model

    def _fit_model(self, X, y, validation_split=0.15, epochs=100, batch_size=128, **kwargs):
        callbacks = [
            EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
            ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5),
        ]
        self.__history = self._model.fit(X, y, validation_split=validation_split, epochs=epochs,
                                         batch_size=batch_size, callbacks=callbacks)

    def _predict_raw(self, X):
        return self._model.predict(X, verbose=0)

    def summary(self):
        self._model.summary()

    def save(self, path="lstm_model.keras"):
        self._model.save(path)

## 5. Run the experiment through the abstract interface

In [6]:
SeedManager.set_seed(42)

In [7]:
loader = CMAPSSDataLoader(subset="FD001")
train_df, test_df, test_last = loader.load()

runner = ExperimentRunner(SequencePreprocessor(rul_cap=125, window=27, seq_length=30),
                          LSTMRULModel(rul_cap=125, seq_length=30, units=64, learning_rate=1e-3))
print(runner)
results = runner.run(train_df, test_df, test_last,
                     validation_split=0.15, epochs=100, batch_size=128)

ExperimentRunner(SequencePreprocessor(rul_cap=125, window=27), LSTMRULModel(name='LSTM', not fitted))


2026-10-24 22:31:46.134931: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M5
2026-10-24 22:31:46.134988: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2026-10-24 22:31:46.134994: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 12.48 GB
2026-10-24 22:31:46.135174: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-10-24 22:31:46.135192: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Epoch 1/100


2026-10-24 22:31:46.987424: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


118/118 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - loss: 6559.0264 - mae: 70.1530 - r2_score: -2.7698 - val_loss: 5940.4663 - val_mae: 66.2095 - val_r2_score: -2.3230 - learning_rate: 0.0010
Epoch 2/100
118/118 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 5263.9595 - mae: 61.9831 - r2_score: -2.0255 - val_loss: 4992.9082 - val_mae: 60.2911 - val_r2_score: -1.7929 - learning_rate: 0.0010
Epoch 3/100
118/118 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - loss: 4432.6904 - mae: 56.6410 - r2_score: -1.5477 - val_loss: 4241.3164 - val_mae: 55.5331 - val_r2_score: -1.3725 - learning_rate: 0.0010
Epoch 4/100
118/118 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - loss: 3777.7729 - mae: 52.3676 - r2_score: -1.1713 - val_loss: 3645.2864 - val_mae: 51.7000 - val_r2_score: -1.0391 - learning_rate: 0.0010
Epoch 5/100
118/118 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - loss: 3259.6709 - mae: 48.9129 - r2_score: -0.8735 - val_loss: 3172.9041 - val_mae: 48.6040 - val_r2_score: -0.7749 - learning_rate: 0.0010
Epoch 6/100
118/118 ━━━━━━━━━━━━

In [8]:
d = runner.data
DataValidator.report(X_train_seq=d["X_train"], X_test_seq=d["X_test"], y_train_seq=d["y_train"])

X_train_seq: shape=(17731, 30, 48), NaNs=0, infs=0, range=(0.000, 1.000)
X_test_seq: shape=(100, 30, 48), NaNs=0, infs=0, range=(0.000, 1.490)
y_train_seq: shape=(17731,), NaNs=0, infs=0, range=(0.000, 125.000)


## 6. Evaluation

In [9]:
print(results.round(4))
print(f"NASA score: {runner.nasa_score():.2f}")

             MAE       MSE     RMSE      R2
Training  4.8271   51.4320   7.1716  0.9706
Testing   9.7439  181.9285  13.4881  0.8946
NASA score: 299.75


In [10]:
lstm = runner.model
lstm.summary()
lstm.save("lstm_model.keras")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        28,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 86,981 (339.77 KB)

 Trainable params: 28,993 (113.25 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 57,988 (226.52 KB)